In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LassoCV
from sklearn.linear_model import Lasso


import classification_rf

from importlib import reload
reload(classification_rf)

from classification_rf import RandomForestClassifier621

In [2]:
def generate_study(features, n, study_label, similarity, intercept_sd):
    X = np.random.normal(loc = 0, scale = 1, size = (n,features)) #Base random inputs

    base = np.arange(features, dtype=float)
    noise = np.random.normal(loc=0, scale=np.std(base), size=features) #inter study weights will vary by this noise parameter controlled by tau
    base  /= np.linalg.norm(base)
    noise /= np.linalg.norm(noise)

    b = similarity * base + (1-similarity)*noise #At similarity 0,1 we get noise and unison.

    weighted_X = np.dot(X,b)
    
    intercept = np.random.normal(0, intercept_sd)

    pX = np.exp(weighted_X+intercept)/(1+np.exp(weighted_X+intercept)) # interval mapping (-inf,inf) to (0,1)

    y = np.random.binomial(n = 1, p = pX) #Random 0 or 1 with probability pX
    
    #print(f"norm of b: {np.linalg.norm(b):.4f}")
    #print(f"X shape: {X.shape}")
    #print(f"Study {study_label}: intercept={intercept:.2f}, base_rate={y.mean():.2f}")
    #print(f"X @ b range: {(X @ b).min():.2f} to {(X @ b).max():.2f}")
    
    labels = [study_label]*n
    X = np.column_stack([X,labels])

    return X,y

def generate_dataset(n_datapoints_per_study, n_studies, features, similarity, intercept_sd):
    
    #Generate first study
    X,y = generate_study(
        features = features, 
        n = n_datapoints_per_study, 
        study_label = 0, 
        similarity = similarity, 
        intercept_sd = intercept_sd
    )

    #Generate studies 2 through n
    for i in range(1,n_studies):
        X_temp,y_temp = generate_study(
            features = features, 
            n = n_datapoints_per_study, 
            study_label = i, 
            similarity = similarity, 
            intercept_sd = intercept_sd
        )
        X = np.concatenate((X,X_temp), axis = 0)
        y = np.concatenate((y,y_temp), axis = 0)

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=24)

    return X_train, X_test, y_train, y_test, X, y

In [4]:
def forest(label_depth, n_estimators, dataset, parameters, seed = 24):
    
    # In forest():
    X_train = dataset.X_train
    X_test  = dataset.X_test
    y_train = dataset.y_train
    y_test = dataset.y_test

    min_samples_leaf = parameters.min_samples_leaf
    max_features = parameters.max_features
    max_depth = parameters.max_depth
    force_label_split = parameters.force_label_split
    oob_score = parameters.oob_score

    np.random.seed(seed)
    
    rf1 = RandomForestClassifier621(
        n_estimators=n_estimators, 
        label_depth=label_depth, 
        min_samples_leaf=min_samples_leaf, 
        max_features=max_features, 
        max_depth=max_depth, 
        force_label_split=force_label_split, 
        oob_score=oob_score
    )
    # Fit the model to the training data
    rf1.fit(X_train, y_train)

    # Calculate the accuracy score for the predictions
    accuracy = rf1.score(X_test, y_test)
    #print("\nTest Accuracy:",accuracy)

    #calculate accuracy for each dataset in Xtest and place into array sourced score
    unique_vals = np.unique(X_test[:, -1])  # get unique values in last column
    source_score = []
    for val in unique_vals:
        sourced_X = X_test[X_test[:, -1] == val]
        sourced_y = y_test[X_test[:, -1] == val]
        source_score.append(rf1.score(sourced_X, sourced_y))

    source_size = [] #size of each source
    for val in unique_vals:
        source_size.append(len(X_train[X_train[:, -1] == val]))

    return accuracy, rf1, source_score, source_size

In [5]:
def forest_wrapper(n_estimators, depth, dataset, parameters, seed = 24):
    #mask = X_test[:,-1] == 3
    #print(f"Seed {seed}: Source 3 test indices = {np.where(mask)[0][:5]}")

    accuracy_score = []
    rf = []
    source_score = []

    for i in range(depth):
        #print("forest", i)
        temp_accuracy_score, temp_rf, temp_source_score, source_size = forest(i, n_estimators, dataset = dataset, parameters=parameters, seed = seed)
        accuracy_score.append(temp_accuracy_score)
        rf.append(temp_rf)
        source_score.append(temp_source_score)
        
        #preds = temp_rf.predict(X_test[mask])
         
        #print(f"Depth {i} Seed {seed}: preds={preds}, true={y_test[mask]}")
    
    source_score = np.array(source_score)
    

    return accuracy_score, rf, source_score, source_size

In [6]:
def errorbars_depth_chart(n_seeds, n_estimators, depth, dataset, parameters, seed = 24):
    forests = [] # forest[0] is forests with same dataset
    accuracy_score = []
    source_score = [] #[seed,depth,source]
    source_size = []

    for i in range(n_seeds):
        temp_accuracy_score, rf, temp_source_score, temp_source_size = forest_wrapper(n_estimators = n_estimators, depth = depth, dataset = dataset, parameters=parameters, seed = seed + i)
        forests.append(rf)
        accuracy_score.append(temp_accuracy_score)
        source_score.append(temp_source_score)
        source_size.append(temp_source_size)
        print(f"Done with Forest {i}")
    
    accuracy_mean = np.mean(accuracy_score, axis=0)  # shape: (n_depths,)
    accuracy_std  = np.std(accuracy_score, axis=0)   # shape: (n_depths,)


    depth_range = np.arange(depth)
    plt.errorbar(depth_range, accuracy_mean, yerr = accuracy_std)
    plt.xlabel('Label Depth')
    plt.ylabel('Accuracy (R^2)')

    return forests, accuracy_score, source_score, source_size

In [8]:
from collections import namedtuple
Dataset = namedtuple('Dataset', ['X_train', 'X_test', 'y_train', 'y_test'])
Params = namedtuple('Params', ['min_samples_leaf', 'max_features', 'max_depth', 'force_label_split', 'oob_score'])

features = 15
similarity = 0.8 #Controls feature similarity between studies
intercept_sd = 2 #Changes the study mean
n = 50
n_studies = 4

X_train, X_test, y_train, y_test, X, y = generate_dataset(
    n_datapoints_per_study = n, 
    n_studies = n_studies, 
    features = features, 
    similarity = similarity, 
    intercept_sd = intercept_sd
)

data = [X,y]

dataset = Dataset(X_train, X_test, y_train, y_test)

parameters = Params(1, "sqrt", 25, False, False) # [min_samples_leaf, max_features, max_depth, force_label_split, oob_score]

forests, accuracy_score, source_score, source_size = errorbars_depth_chart(n_seeds = 10, n_estimators = 200, depth = 20, dataset = dataset, parameters = parameters)

KeyboardInterrupt: 